# Transform Development Notebook

Transform raw bronze data from `s3://is459-crypto-raw-data/bronze/` using PySpark.


## 1. Setup — AWS credentials + SparkSession


In [1]:
import os, multiprocessing, sys
from dotenv import load_dotenv, find_dotenv

# ── 1. Load .env FIRST ────────────────────────────────────────────────────────
dotenv_path = find_dotenv()
print(f".env found at: {dotenv_path}")
load_dotenv(dotenv_path=dotenv_path, override=True)

AWS_KEY = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET = os.environ["AWS_SECRET_ACCESS_KEY"]
AWS_REGION = os.environ["AWS_REGION"]
BUCKET = os.environ["S3_BUCKET_RAW"]
BRONZE_S3 = f"s3a://{BUCKET}/bronze"

# Silver layer — falls back to the raw bucket under a /silver prefix
SILVER_BUCKET = os.environ.get("S3_BUCKET_SILVER", BUCKET)
SILVER_S3 = f"s3a://{SILVER_BUCKET}/silver/ohlcv"

# ClickHouse connection (Python client only — no JDBC)
CH_HOST = os.environ.get("CLICKHOUSE_HOST", "localhost")
CH_PORT = os.environ.get("CLICKHOUSE_PORT", "8123")
CH_DB = os.environ.get("CLICKHOUSE_DB", "default")
CH_USER = os.environ.get("CLICKHOUSE_USER", "default")
CH_PASSWORD = os.environ.get("CLICKHOUSE_PASSWORD", "")

# ── 2. Pin worker Python to the same interpreter as the driver ────────────────
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

NUM_CORES = multiprocessing.cpu_count()  # 11
PARALLELISM = NUM_CORES * 3  # 33
print(f"Python executable : {sys.executable}")
print(f"Logical CPUs      : {NUM_CORES}  →  parallelism = {PARALLELISM}")

# ── 3. JVM / packages — must be set BEFORE importing pyspark ─────────────────
# ClickHouse JDBC removed — ClickHouse reads Parquet from S3 directly (§8).
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 8g "
    "--packages "
    "org.apache.hadoop:hadoop-aws:3.4.1,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

# ── 4. Import pyspark AFTER setting env vars ──────────────────────────────────
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# ── 5. SparkSession ───────────────────────────────────────────────────────────
spark = (
    SparkSession.builder.master("local[*]")
    .appName("transform-dev")
    # ── AWS credentials ───────────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.access.key", AWS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET)
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    # ── S3A connection pool ───────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.connection.maximum", "200")
    .config("spark.hadoop.fs.s3a.threads.max", "200")
    # ── S3A read throughput ───────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.readahead.range", "8388608")  # 8 MB  (was 64 KB)
    .config("spark.hadoop.fs.s3a.block.size", "134217728")  # 128 MB (was 32 MB)
    .config("spark.hadoop.fs.s3a.input.fadvise", "sequential")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer")
    .config("spark.hadoop.fs.s3a.multipart.size", "67108864")  # 64 MB
    # ── JVM / GC ──────────────────────────────────────────────────────────────
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config(
        "spark.driver.extraJavaOptions",
        "-XX:+UseG1GC "
        "-XX:G1HeapRegionSize=16m "
        "-XX:InitiatingHeapOccupancyPercent=35 "
        "-XX:+UseStringDeduplication",
    )
    # ── Serialisation ─────────────────────────────────────────────────────────
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    # ── Memory ────────────────────────────────────────────────────────────────
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.1")  # 90% of pool to execution
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    # ── Parallelism ───────────────────────────────────────────────────────────
    .config("spark.default.parallelism", str(PARALLELISM))  # 33
    .config("spark.sql.shuffle.partitions", str(PARALLELISM))  # 33
    # ── Shuffle I/O ───────────────────────────────────────────────────────────
    .config("spark.shuffle.file.buffer", "1m")
    .config("spark.reducer.maxSizeInFlight", "96m")
    .config("spark.shuffle.localDisk.file.output.buffer", "5m")
    # ── SQL / AQE ─────────────────────────────────────────────────────────────
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "134217728")
    .config("spark.sql.files.maxPartitionBytes", "134217728")
    .config("spark.sql.files.openCostInBytes", "67108864")
    # ── Network timeouts ──────────────────────────────────────────────────────
    .config("spark.network.timeout", "600s")
    .config("spark.sql.broadcastTimeout", "600")
    # ── macOS fixes ───────────────────────────────────────────────────────────
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.executor.processTreeMetrics.enabled", "false")
    .getOrCreate()
)

hadoop_ver = spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
print(f"Spark             : {spark.version}")
print(f"Hadoop            : {hadoop_ver}")
print(f"Bucket (raw)      : {BUCKET}")
print(f"Bucket (silver)   : {SILVER_BUCKET}")
print(f"ClickHouse        : {CH_HOST}:{CH_PORT}/{CH_DB}")
print(f"Default parallel  : {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver mem        : {spark.conf.get('spark.driver.memory')}")
print("SparkSession ready")

# ── 6. boto3 client (listing only) ────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name=AWS_REGION,
)


def list_s3_folders(prefix):
    """Return immediate sub-folder prefixes under a given prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [cp["Prefix"] for cp in resp.get("CommonPrefixes", [])]

.env found at: /Users/lunlun/Downloads/Github/IS459_Crypto_BigData_Pipeline/.env
Python executable : /usr/local/bin/python
Logical CPUs      : 11  →  parallelism = 33


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 20:47:08 WARN Utils: Your hostname, Chengs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.203.17.71 instead (on interface en0)
26/03/26 20:47:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/26 20:47:08 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
26/03/26 20:47:08 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users

Spark             : 4.0.0
Hadoop            : 3.4.1
Bucket (raw)      : is459-crypto-raw-data
Bucket (silver)   : is459-crypto-raw-data
ClickHouse        : localhost:8123/default
Default parallel  : 33
Shuffle partitions: 33
Driver mem        : 8g
SparkSession ready


## 2. Ticker Discovery — `kaggle/` vs `binance/`


In [2]:
# boto3 list_objects — metadata only, no data read
kaggle_tickers = {
    p.rstrip("/").split("/")[-1] for p in list_s3_folders("bronze/kaggle/btc-price-1m/")
}
binance_tickers = {
    p.rstrip("/").split("/")[-1] for p in list_s3_folders("bronze/binance/")
}

in_both = sorted(kaggle_tickers & binance_tickers)
only_kaggle = sorted(kaggle_tickers - binance_tickers)
only_binance = sorted(binance_tickers - kaggle_tickers)

print(f"Kaggle  : {len(kaggle_tickers):>4}  tickers")
print(f"Binance : {len(binance_tickers):>4}  tickers")

print(f"\n── In both ({len(in_both)}) ─────────────────────────────────────────────")
for t in in_both:
    print(f"  {t}")

print(f"\n── Kaggle only ({len(only_kaggle)}) — not in Binance ─────────────────────")
for t in only_kaggle:
    print(f"  {t}")

print(f"\n── Binance only ({len(only_binance)}) — not in Kaggle ────────────────────")
for t in only_binance:
    print(f"  {t}")

Kaggle  :   50  tickers
Binance :   46  tickers

── In both (34) ─────────────────────────────────────────────
  AAVEUSDT
  ADAUSDT
  ALGOUSDT
  ATOMUSDT
  AVAXUSDT
  BATUSDT
  BCHUSDT
  BNBUSDT
  BTCUSDT
  CHZUSDT
  CRVUSDT
  DOGEUSDT
  DOTUSDT
  ENJUSDT
  ETHUSDT
  FILUSDT
  HBARUSDT
  ICPUSDT
  LINKUSDT
  LTCUSDT
  MANAUSDT
  NEARUSDT
  SANDUSDT
  SHIBUSDT
  SOLUSDT
  SUSHIUSDT
  THETAUSDT
  TRXUSDT
  UNIUSDT
  VETUSDT
  XLMUSDT
  XRPUSDT
  XTZUSDT
  ZILUSDT

── Kaggle only (16) — not in Binance ─────────────────────
  CAKEUSDT
  DASHUSDT
  DEXEUSDT
  ENAUSDT
  EOSUSDT
  ETCUSDT
  GRTUSDT
  IOSTUSDT
  IOTAUSDT
  NEOUSDT
  QTUMUSDT
  RVNUSDT
  STMXUSDT
  TFUELUSDT
  TIAUSDT
  ZECUSDT

── Binance only (12) — not in Kaggle ────────────────────
  1INCHUSDT
  APTUSDT
  AXSUSDT
  COMPUSDT
  EGLDUSDT
  FLOWUSDT
  GALAUSDT
  LRCUSDT
  QNTUSDT
  RUNEUSDT
  SNXUSDT
  YFIUSDT


## 3. `kaggle/` Preprocessing

`symbol` column is derived from the S3 folder name.


### 3.1 Standardizing `kaggle` dataframe


In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampType,
)

KAGGLE_SCHEMA = StructType(
    [
        StructField("timestamp", TimestampType(), True),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField("close_time", TimestampType(), True),
        StructField("quote_asset_volume", DoubleType(), True),
        StructField("number_of_trades", LongType(), True),
        StructField("taker_buy_base_asset_volume", DoubleType(), True),
        StructField("taker_buy_quote_asset_volume", DoubleType(), True),
        StructField("ignore", StringType(), True),
    ]
)

ORDERED_COLS = [
    "timestamp",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume",
]


### 3.2 Compiled Kaggle DataFrame

`kaggle_raw` — explicit schema, single-pass CSV read across all ticker folders.


In [4]:
kaggle_paths = [
    f"s3a://{BUCKET}/{p}" for p in list_s3_folders("bronze/kaggle/btc-price-1m/")
]

kaggle_raw = (
    spark.read.schema(KAGGLE_SCHEMA)
    .csv(kaggle_paths, header=True)
    .drop("ignore")
    .withColumn(
        "symbol", F.regexp_extract(F.input_file_name(), r"/btc-price-1m/([^/]+)/", 1)
    )
    .select(ORDERED_COLS)
)

26/03/26 20:47:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## 4. `binance/` Preprocessing

### 4.1 Compiled Binance DataFrame

`binance_raw` — explicit schema skips JSON schema-inference (avoids a second full S3 scan). Ticker folders resolved via boto3; passed as an explicit list so S3A does not attempt wildcard expansion.


In [5]:
BINANCE_SCHEMA = StructType(
    [
        StructField("close", StringType(), True),
        StructField("close_time", LongType(), True),
        StructField("high", StringType(), True),
        StructField("low", StringType(), True),
        StructField("number_of_trades", LongType(), True),
        StructField("open", StringType(), True),
        StructField("quote_asset_volume", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("taker_buy_base_asset_volume", StringType(), True),
        StructField("taker_buy_quote_asset_volume", StringType(), True),
        StructField("timestamp", LongType(), True),
        StructField("volume", StringType(), True),
    ]
)

# S3A does not expand glob wildcards — resolve paths explicitly via boto3
binance_paths = [f"s3a://{BUCKET}/{p}" for p in list_s3_folders("bronze/binance/")]

binance_raw = (
    spark.read.schema(BINANCE_SCHEMA)
    .option("mode", "DROPMALFORMED")
    .json(binance_paths)
    .select(
        (F.col("timestamp") / 1000).cast("timestamp").alias("timestamp"),
        F.col("symbol"),
        F.col("open").cast(DoubleType()),
        F.col("high").cast(DoubleType()),
        F.col("low").cast(DoubleType()),
        F.col("close").cast(DoubleType()),
        F.col("volume").cast(DoubleType()),
        (F.col("close_time") / 1000).cast("timestamp").alias("close_time"),
        F.col("quote_asset_volume").cast(DoubleType()),
        F.col("number_of_trades"),
        F.col("taker_buy_base_asset_volume").cast(DoubleType()),
        F.col("taker_buy_quote_asset_volume").cast(DoubleType()),
    )
    .select(ORDERED_COLS)
)

## 5. Union — Kaggle ∪ Binance

Both DataFrames share `ORDERED_COLS` — `unionByName` is schema-safe and avoids positional mismatches. No cache needed: `combined_df` is the terminal result, materialised once by the downstream write.


In [6]:
combined_df = kaggle_raw.unionByName(binance_raw)

## 6. Data Quality

**4 actions, 2 S3 scans** — `no_null_df` and `final_df` are both persisted `DISK_ONLY` so each subsequent count/show is a disk cache hit, not an S3 re-read.

### 6.1 Trim string columns


In [7]:
# No cache — full pre-clean dataset is too large for in-memory storage
trimmed_df = combined_df.withColumn("symbol", F.trim(F.col("symbol")))

### 6.2 Drop nulls


In [8]:
from pyspark import StorageLevel

no_null_df = trimmed_df.dropna()

### 6.3 Drop duplicates


In [9]:
final_df = no_null_df.dropDuplicates(["timestamp", "symbol"])

### 6.4 Final output

Action 4 — `show()` served from `final_df` disk cache, no S3 re-read.


In [ ]:
final_df.show(20, truncate=False)

26/03/26 20:47:41 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
26/03/26 20:47:50 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
26/03/26 20:47:52 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
26/03/26 20:48:02 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has been deprecated as of Spark 4.0.0 and may be removed in the future. Please use spark.shuffle.localDisk.file.output.buffer
26/03/26 20:48:09 WARN SparkConf: The configuration key 'spark.shuffle.unsafe.file.output.buffer' has be

Py4JJavaError: An error occurred while calling o246.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file s3a://is459-crypto-raw-data/bronze/kaggle/btc-price-1m/UNIUSDT/UNIUSDT.csv.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:856)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:583)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluator

26/03/26 21:28:47 WARN TaskSetManager: Lost task 103.0 in stage 2.0 (TID 199) (10.203.17.71 executor driver): TaskKilled (Stage cancelled: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file s3a://is459-crypto-raw-data/bronze/kaggle/btc-price-1m/UNIUSDT/UNIUSDT.csv.  SQLSTATE: KD001)
26/03/26 21:28:47 WARN TaskSetManager: Lost task 88.0 in stage 2.0 (TID 184) (10.203.17.71 executor driver): TaskKilled (Stage cancelled: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file s3a://is459-crypto-raw-data/bronze/kaggle/btc-price-1m/UNIUSDT/UNIUSDT.csv.  SQLSTATE: KD001)
26/03/26 21:28:47 WARN TaskSetManager: Lost task 107.0 in stage 2.0 (TID 203) (10.203.17.71 executor driver): TaskKilled (Stage cancelled: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file s3a://is459-crypto-raw-data/bronze/kaggle/btc-price-1m/UNIUSDT/UNIUSDT.csv.  SQLSTATE: KD001)
26/03/26 21:36:15 WARN TaskSetManager: Lost task 102.0 in stage 2.0 (TID 198) (10.203.17.71 executor driver): 

## 7. Write to S3 Silver — Parquet

**Partitioning strategy: `symbol` only.**

Dashboard queries almost always filter on a single symbol + time range. Partitioning by `symbol` gives:

- One directory per ticker → full partition pruning when a query filters `WHERE symbol = 'BTCUSDT'`
- ~1.75 M rows / ~80 MB compressed per file — well within Parquet's sweet spot

`sortWithinPartitions("timestamp")` orders rows inside each Parquet file without a global shuffle. Parquet readers (ClickHouse S3, Athena, etc.) use the min/max row-group statistics to skip entire row groups on time-range predicates.

A global `orderBy` is **not** used — it would force a single-partition sort of 168 M rows with no benefit over per-partition sorting for query performance.

### 7.1 Write Parquet


In [ ]:
(
    final_df.repartition(F.col("symbol"))  # one shuffle partition per ticker
    .sortWithinPartitions("timestamp")  # ordered within file, no global sort
    .write.partitionBy("symbol")  # s3://.../silver/ohlcv/symbol=BTCUSDT/
    .mode("overwrite")
    .option("compression", "snappy")  # fast compress/decompress, good ratio
    .parquet(SILVER_S3)
)
print(f"Parquet written → {SILVER_S3}")